# Imports and setup

In [3]:
import time

import pandas as pd

from demand.data import clean, load_subset
from demand.features import build_features
from demand.metrics import evaluate
from demand.nets import fit_net, predict_net, select_windows
from demand.sequences import make_windows
from demand.validation import split_holdout, walk_forward_splits

features = build_features(clean(load_subset()))
windows = make_windows(features)
train_part, holdout = split_holdout(features)

# Walk-forward for RNN and LSTM

In [4]:
def run_walk_forward_net(cell, **params):
    rows = []
    for fold, (train_dates, _) in enumerate(walk_forward_splits(train_part.date), 1):
        train_last = train_dates[-1].to_datetime64()
        train_mask, stop_mask, valid_mask = select_windows(windows, train_last)

        start = time.perf_counter()
        net, history = fit_net(windows, train_mask, stop_mask, cell=cell, **params)
        fit_seconds = time.perf_counter() - start

        pred = predict_net(net, windows, valid_mask)
        actual = windows["y"][valid_mask] * windows["level"][valid_mask][:, None]
        open_days = windows["mask"][valid_mask]
        rows.append({
            "fold": fold, **evaluate(actual[open_days], pred[open_days]),
            "epochs": len(history), "fit_s": fit_seconds,
        })
    return pd.DataFrame(rows).set_index("fold")

rnn = run_walk_forward_net("rnn")
lstm = run_walk_forward_net("lstm")
for name, result in (("rnn", rnn), ("lstm", lstm)):
    display(name)
    display(result.round(3))
    display(result.mean().round(3))

'rnn'

,MAE,RMSE,WAPE,epochs,fit_s
fold,,,,,
1,756.943,1314.916,0.135,43,55.275
2,750.072,1259.703,0.153,32,46.048
3,645.956,1045.796,0.137,25,38.682
4,599.658,1100.887,0.123,20,31.698
5,575.265,1078.000,0.123,28,47.712
6,567.204,996.461,0.118,56,98.556
7,509.629,867.414,0.105,30,53.400
8,483.974,803.345,0.103,17,31.505


MAE        611.088
RMSE      1058.315
WAPE         0.125
epochs      31.375
fit_s       50.360
dtype: float64

'lstm'

,MAE,RMSE,WAPE,epochs,fit_s
fold,,,,,
1,874.883,1393.523,0.156,27,34.163
2,458.802,753.707,0.093,38,57.473
3,642.194,1031.823,0.136,42,67.378
4,632.929,1115.651,0.130,25,42.875
5,599.636,1090.162,0.128,19,35.405
6,639.678,1083.128,0.133,31,60.147
7,576.452,930.986,0.119,39,78.477
8,507.970,848.093,0.108,16,33.600


MAE        616.568
RMSE      1030.884
WAPE         0.125
epochs      29.625
fit_s       51.190
dtype: float64

# Other random seeds

In [5]:
def run_seeds(cell, seeds=(1, 2), **params):
    results = {seed: run_walk_forward_net(cell, seed=seed, **params).mean() for seed in seeds}
    return pd.DataFrame(results).T.round(3)

display(run_seeds("rnn"))
display(run_seeds("lstm"))

,MAE,RMSE,WAPE,epochs,fit_s
1,602.792,1040.004,0.123,33.50,50.340
2,593.847,1026.291,0.121,39.25,57.737


,MAE,RMSE,WAPE,epochs,fit_s
1,605.897,1009.236,0.123,31.125,52.946
2,601.074,1003.203,0.122,32.875,55.932


# Conclusions

- Setup: one model for all 25 series that reads the last 56 days (sales divided by the series level plus calendar features), takes the known features of the next 28 days and a series embedding, and returns all 28 days at once. The same 8 walk-forward folds and the same metrics on open days as for the other models. The simple RNN and the LSTM differ only in the recurrent cell.
- Training: AdamW with weight decay, dropout 0.2, L1 loss on open days, early stopping on 14 windows whose targets do not overlap with the training windows (about 30 epochs per fold).
- Both networks beat the seasonal median baseline (WAPE 0.123 against 0.141) and lose clearly to the boosters (XGBoost with the scaled target: 0.105).
- RNN and LSTM are on the same level: WAPE 0.123 for both (mean of 3 seeds, spread 0.121-0.125). The LSTM has about 3% lower RMSE but no gain on MAE or WAPE, so the gates do not help on a 56-day window.
- The gap to the boosters (0.018) is much larger than the noise between seeds (0.004), so the conclusion is stable.
- Cost: about 50 s of training per fold against about 1 s for XGBoost, with a worse result.
- The networks see the history up to the last training day, while the boosters use lags of at least 28 days. The networks lose even with this advantage.
- Limits: the window length, hidden size and dropout were not tuned, the early stopping set is small (350 overlapping windows) and noisy, and results come from CPU runs with 3 seeds.